# Load in and clean data

## Loading in Packages and Data

In [0]:
%pip install "matplotlib==3.8.4" matplotlib-venn
%pip install upsetplot
dbutils.library.restartPython()

In [0]:
import pandas as pd

results_GPT_full_df = pd.read_json(("/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full_final.json"), orient="records")
display(results_GPT_full_df)
print(len(results_GPT_full_df))

results_mini_full_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_GPT_Mini_Output_Results_full_final.json", orient="records")
display(results_mini_full_df)
print(len(results_mini_full_df))

results_claude_full_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_final.json", orient="records")
display(results_claude_full_df)
print(len(results_claude_full_df))

manual_score = pd.read_csv("/Workspace/Users/user/user/manual_score.csv")
display(manual_score)

notes_enrolled4 = pd.read_csv("/Workspace/Users/user/user/notes_enrolled4.csv")
display(notes_enrolled4)
print(len(notes_enrolled4))

checklist_filtered = pd.read_csv("/Workspace/Users/user/user/checklist_filtered.csv")
display(checklist_filtered)

## Data Cleaning

In [0]:
# list of notes that are in notes_enrolled4 but were not in the manual score. These notes are kept in the analysis. 
gpt_not_manual_ids = set(results_GPT_full_df['Subject ID']) - set(manual_score['Subject ID'])
display(notes_enrolled4[notes_enrolled4['Subject ID'].isin(gpt_not_manual_ids)])
len(gpt_not_manual_ids)

In [0]:
# Printing details about those in notes_enrolled4 but not in manual_score as well as in GPT results but not in manual_score

in_notes_not_manual = set(notes_enrolled4['Subject ID']) - set(manual_score['Subject ID'])
in_gpt_not_manual = set(results_GPT_full_df['Subject ID']) - set(manual_score['Subject ID'])

print(f"In notes_enrolled4 but not manual_score: {len(in_notes_not_manual)}")
print(f"In GPT results but not manual_score: {len(in_gpt_not_manual)}")
print(f"GPT-not-manual is subset of notes-not-manual: {in_gpt_not_manual.issubset(in_notes_not_manual)}")
print(in_notes_not_manual)

In [0]:
#Loading in redcap mapping
REDCap_mapping_full_path = "/Workspace/Users/user/user/data_labels_filtered4.csv"

REDCap_mapping = pd.read_csv(REDCap_mapping_full_path)
print('Number of rows:', len(REDCap_mapping))
print('Number of columns:', len(REDCap_mapping.columns))
display(REDCap_mapping)

In [0]:
# Check if the notes missing in the manual review appear in REDCap_mapping or in checklist_filtered
missing_in_redcap = set(in_notes_not_manual) - set(REDCap_mapping.values.flatten())
missing_in_checklist = set(in_notes_not_manual) - missing_in_redcap

print(f"Not in REDCap_mapping at all: {len(missing_in_redcap)}")
print(missing_in_redcap)
print(f"\nIn REDCap_mapping but phenotypes didn't match checklist_filtered: {len(missing_in_checklist)}")
print(missing_in_checklist)

In [0]:
# For those in REDCap_mapping, what phenotypes did they have?
# Recreate records from REDCap_mapping
records = []
for i in range(0, len(REDCap_mapping)):
    phen = []
    for j in range(0, len(REDCap_mapping.columns)):
        if pd.isnull(REDCap_mapping.iloc[i][j]) == False:
            phen.append(REDCap_mapping.iloc[i][j])
    records.append(phen)

# Now check phenotypes for the 11 missing Subject IDs
for sid in missing_in_checklist:
    sid_row = [i for i, record in enumerate(records) if record and record[0] == sid]
    if sid_row:
        phenotypes = [x for x in records[sid_row[0]] if not str(x).startswith('WES')]
        print(f"\n{sid}: {phenotypes}")
    else:
        print(f"\n{sid}: not found in records")

In [0]:
print(f"notes_enrolled4 subjects: {notes_enrolled4['Subject ID'].nunique()}")
print(f"results_GPT_full_df subjects: {results_GPT_full_df['Subject ID'].nunique()}")
print(f"results_mini_full_df subjects: {results_mini_full_df['Subject ID'].nunique()}")
print(f"results_claude_full_df subjects: {results_claude_full_df['Subject ID'].nunique()}")
print(f"manual_score subjects: {manual_score['Subject ID'].nunique()}")

In [0]:
# Finding phenotypes that are not represented in gpt, claude, mini, and manual

phenotypes_results_gpt = set(results_GPT_full_df['Phenotype'].dropna().unique())
phenotypes_results_mini = set(results_mini_full_df['Phenotype'].dropna().unique())
phenotypes_results_claude = set(results_claude_full_df['Phenotype'].dropna().unique())

phenotypes_manual = set(manual_score['Phenotype'].dropna().unique())

missing_from_manual = sorted(phenotypes_results_gpt - phenotypes_manual)
missing_from_gpt = sorted(phenotypes_manual - phenotypes_results_gpt)
missing_from_mini = sorted(phenotypes_manual - phenotypes_results_mini)
missing_from_claude = sorted(phenotypes_manual - phenotypes_results_claude)

print(len(missing_from_manual), "values in GPT but not in manual_score")
print(missing_from_manual)

print()

print(len(missing_from_gpt), "values in manual_score but not in GPT")
print(missing_from_gpt)

print()

print(len(missing_from_mini), "values in manual_score but not in mini")
print(missing_from_mini)

print()

print(len(missing_from_claude), "values in manual_score but not in claude")
print(missing_from_claude)

In [0]:
# Remapping phenotypes such that any parenthesis/spacing issues are normalized and Chairi I malformation (which claude hallucinated and is a subset of the original phenotype that was input into the model) is renamed to chiari malformation, the original name

remap = {
    # Parenthesis fixes — all should have closing )
    'Double Outlet Right Ventricle (DORV': 'Double Outlet Right Ventricle (DORV)',
    'Hypertrophic cardiomyopathy (HCM': 'Hypertrophic cardiomyopathy (HCM)',
    'Tetralogy of Fallot (TOF': 'Tetralogy of Fallot (TOF)',
    'CPEO (opthalmoplegia': 'CPEO (opthalmoplegia)',
    'Autism Spectrum Disorder (ASD': 'Autism Spectrum Disorder (ASD)',
    'Diabetes Mellitus Type I (insulin dependent': 'Diabetes Mellitus Type I (insulin dependent)',
    'Diabetes Mellitus Type II (non-insulin dependent': 'Diabetes Mellitus Type II (non-insulin dependent)',
    # Hallucination fix
    'Chiari I malformation': 'Chiari malformation',
}

manual_score['Phenotype'] = manual_score['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
results_GPT_full_df['Phenotype'] = results_GPT_full_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
results_mini_full_df['Phenotype'] = results_mini_full_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
results_claude_full_df['Phenotype'] = results_claude_full_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
checklist_filtered['Term from checklist'] = checklist_filtered['Term from checklist'].str.strip().map(lambda x: remap.get(x, x))

In [0]:
# Showing how many duplicated phenotypes remain
results_GPT_full_df.duplicated(subset=['Subject ID', 'Phenotype']).sum()

In [0]:
# Finding phenotypes that are not represented in gpt, claude, mini, and manual after deduplication

phenotypes_results_gpt = set(results_GPT_full_df['Phenotype'].dropna().unique())
phenotypes_results_mini = set(results_mini_full_df['Phenotype'].dropna().unique())
phenotypes_results_claude = set(results_claude_full_df['Phenotype'].dropna().unique())

phenotypes_manual = set(manual_score['Phenotype'].dropna().unique())

missing_from_manual = sorted(phenotypes_results_gpt - phenotypes_manual)
missing_from_gpt = sorted(phenotypes_manual - phenotypes_results_gpt)
missing_from_mini = sorted(phenotypes_manual - phenotypes_results_mini)
missing_from_claude = sorted(phenotypes_manual - phenotypes_results_claude)

print(len(missing_from_manual), "values in results_GPT_full_df but not in manual_score")
print(missing_from_manual)

print()

print(len(missing_from_gpt), "values in manual_score but not in GPT")
print(missing_from_gpt)

print()

print(len(missing_from_mini), "values in manual_score but not in mini")
print(missing_from_mini)

print()

print(len(missing_from_claude), "values in manual_score but not in claude")
print(missing_from_claude)

In [0]:
#Renaming columns
results_GPT_full_df = results_GPT_full_df.rename(columns={'Score': 'GPT_Score'})
results_mini_full_df = results_mini_full_df.rename(columns={'Score': 'GPT_Mini_Score'})
results_claude_full_df = results_claude_full_df.rename(columns={'Score': 'Claude_Sonnet_Score'})

In [0]:
# Viewing score totals across entire dataframes (no subsetting)

score_totals = {}

for name, df, score_col, agg in [
    ('Manual', manual_score, 'Phenotype', 'count'),
    ('GPT', results_GPT_full_df, 'GPT_Score', 'sum'),
    ('GPT Mini', results_mini_full_df, 'GPT_Mini_Score', 'sum'),
    ('Claude Sonnet', results_claude_full_df, 'Claude_Sonnet_Score', 'sum'),
]:
    if agg == 'count':
        score_totals[name] = df[score_col].count()
    else:
        score_totals[name] = df[score_col].sum()

print("Total phenotypes scored positive, per model, across all subjects:")
for model, total in score_totals.items():
    print(f"  {model}: {total}")

In [0]:
# Displaying notes under 150 characters long 
short_notes = notes_enrolled4[notes_enrolled4['NOTE_TEXT'].str.len() < 150]
display(short_notes)

In [0]:
# Viewing score totals for rows to be dropped in ids_to_drop

score_totals = {}

for name, df, score_col, agg in [
    ('Manual', manual_score, 'Phenotype', 'count'),
    ('GPT', results_GPT_full_df, 'GPT_Score', 'sum'),
    ('GPT Mini', results_mini_full_df, 'GPT_Mini_Score', 'sum'),
    ('Claude Sonnet', results_claude_full_df, 'Claude_Sonnet_Score', 'sum'),
]:
    subset = df[df['Subject ID'].isin(short_notes['Subject ID'])]
    if agg == 'count':
        score_totals[name] = subset[score_col].count()
    else:
        score_totals[name] = subset[score_col].sum()

print(f"Subjects with notes under 150 characters: {short_notes['Subject ID'].nunique()}")
print("\nTotal phenotypes scored positive, per model, for these subjects:")
for model, total in score_totals.items():
    print(f"  {model}: {total}")

In [0]:
# Displayign the phenotypes that the manual had scored to these short notes
display(manual_score[manual_score['Subject ID'].isin(short_notes['Subject ID'])])

In [0]:
# Dropping notes under 150 characters long
ids_to_drop = short_notes['Subject ID']

manual_score = manual_score[~manual_score['Subject ID'].isin(ids_to_drop)]
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin(ids_to_drop)]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin(ids_to_drop)]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin(ids_to_drop)]

In [0]:
# Subjects where GPT scored 0 phenotypes total
gpt_totals = results_GPT_full_df.groupby('Subject ID')['GPT_Score'].sum()
zero_score_subjects = gpt_totals[gpt_totals == 0].index.tolist()

print(f"Subjects with 0 phenotypes scored by GPT: {len(zero_score_subjects)}")
print(zero_score_subjects)

# Pull their notes from notes_enrolled4
zero_score_notes = notes_enrolled4[notes_enrolled4['Subject ID'].isin(zero_score_subjects)]

print(f"\nMatching note rows found: {len(zero_score_notes)}")
display(zero_score_notes[['Subject ID', 'NOTE_TEXT', 'NOTE_TYPE']])

In [0]:
# Viewing score totals for rows dropped in next cell
subject_ids = [8 WES IDs]

score_totals = {}

for name, df, score_col, agg in [
    ('Manual', manual_score, 'Phenotype', 'count'),
    ('GPT', results_GPT_full_df, 'GPT_Score', 'sum'),
    ('GPT Mini', results_mini_full_df, 'GPT_Mini_Score', 'sum'),
    ('Claude Sonnet', results_claude_full_df, 'Claude_Sonnet_Score', 'sum'),
]:
    subset = df[df['Subject ID'].isin(subject_ids)]
    if agg == 'count':
        score_totals[name] = subset[score_col].count()
    else:
        score_totals[name] = subset[score_col].sum()

print(f"Subjects to be dropped: {len(subject_ids)}")
print("\nTotal phenotypes scored positive, per model, for these subjects:")
for model, total in score_totals.items():
    print(f"  {model}: {total}")

In [0]:
manual_score = manual_score[~manual_score['Subject ID'].isin([9 WES IDs])]
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin([9 WES IDs])]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin([9 WES IDs])]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin([9 WES IDs])]

In [0]:
# Subjects where mini scored 0 phenotypes total
mini_totals = results_mini_full_df.groupby('Subject ID')['GPT_Mini_Score'].sum()
zero_score_subjects = mini_totals[mini_totals == 0].index.tolist()

print(f"Subjects with 0 phenotypes scored by mini: {len(zero_score_subjects)}")
print(zero_score_subjects)

# Pull their notes from notes_enrolled4
zero_score_notes = notes_enrolled4[notes_enrolled4['Subject ID'].isin(zero_score_subjects)]

print(f"\nMatching note rows found: {len(zero_score_notes)}")
display(zero_score_notes[['Subject ID', 'NOTE_TEXT']])

In [0]:
# Subjects where claude scored 0 phenotypes total
claude_totals = results_claude_full_df.groupby('Subject ID')['Claude_Sonnet_Score'].sum()
zero_score_subjects = claude_totals[claude_totals == 0].index.tolist()

print(f"Subjects with 0 phenotypes scored by claude: {len(zero_score_subjects)}")
print(zero_score_subjects)

# Pull their notes from notes_enrolled4
zero_score_notes = notes_enrolled4[notes_enrolled4['Subject ID'].isin(zero_score_subjects)]

print(f"\nMatching note rows found: {len(zero_score_notes)}")
display(zero_score_notes[['Subject ID', 'NOTE_TEXT']])

In [0]:
display(notes_enrolled4[notes_enrolled4['Subject ID'] == 1 WES ID])
display(manual_score[manual_score['Subject ID'] == 1 WES ID])
display(results_GPT_full_df[results_GPT_full_df['Subject ID'] == 1 WES ID])
display(results_mini_full_df[results_mini_full_df['Subject ID'] == 1 WES ID])
display(results_claude_full_df[results_claude_full_df['Subject ID'] == 1 WES ID])

In [0]:
matches = notes_enrolled4[notes_enrolled4['NOTE_TEXT'].str.contains('consent forms', case=True, na=False)]

print(f"Rows matching 'consent forms' (case-sensitive): {len(matches)}")
display(matches)

In [0]:
# Check that results_GPT_full_df, results_mini_full_df, and results_claude_full_df
# all have the exact same set of Subject IDs (no extras, no missing), and that
# all of those Subject IDs are also present in manual_score.

gpt_ids = set(results_GPT_full_df['Subject ID'].unique())
mini_ids = set(results_mini_full_df['Subject ID'].unique())
claude_ids = set(results_claude_full_df['Subject ID'].unique())
manual_ids = set(manual_score['Subject ID'].unique())

print(f"GPT subjects: {len(gpt_ids)}")
print(f"Mini subjects: {len(mini_ids)}")
print(f"Claude subjects: {len(claude_ids)}")
print(f"Manual subjects: {len(manual_ids)}")
print()

# --- Pairwise comparison across the three result sets ---
all_three_equal = (gpt_ids == mini_ids == claude_ids)
print(f"GPT, Mini, and Claude all have identical Subject ID sets: {all_three_equal}")
print()

if not all_three_equal:
    print("GPT vs Mini:")
    print(f"  In GPT but not Mini: {sorted(gpt_ids - mini_ids)}")
    print(f"  In Mini but not GPT: {sorted(mini_ids - gpt_ids)}")
    print()
    print("GPT vs Claude:")
    print(f"  In GPT but not Claude: {sorted(gpt_ids - claude_ids)}")
    print(f"  In Claude but not GPT: {sorted(claude_ids - gpt_ids)}")
    print()
    print("Mini vs Claude:")
    print(f"  In Mini but not Claude: {sorted(mini_ids - claude_ids)}")
    print(f"  In Claude but not Mini: {sorted(claude_ids - mini_ids)}")
    print()

# --- Check all three are fully contained in manual_score ---
combined_result_ids = gpt_ids | mini_ids | claude_ids
missing_from_manual = combined_result_ids - manual_ids

print(f"All GPT/Mini/Claude subjects present in manual_score: {len(missing_from_manual) == 0}")
if missing_from_manual:
    print(f"  Subjects missing from manual_score: {sorted(missing_from_manual)}")

# --- Optional: also flag if manual_score has extra subjects not in any result set ---
extra_in_manual = manual_ids - combined_result_ids
if extra_in_manual:
    print(f"\nSubjects in manual_score but not in any of GPT/Mini/Claude: {len(extra_in_manual)}")
    print(f"  {sorted(extra_in_manual)}")

In [0]:
# View the notes for GPT/Mini/Claude subjects not present in manual_score

subject_ids_to_check = combined_result_ids - manual_ids

notes_subset = notes_enrolled4[
    notes_enrolled4['Subject ID'].isin(subject_ids_to_check)
][['Subject ID', 'NOTE_FILE_TIME_DTTM', 'NOTE_TEXT', 'Enrollment Status', 'Date of last chart review:']].dropna(subset=['NOTE_TEXT']).reset_index(drop=True)

display(notes_subset)

In [0]:
import pandas as pd

output_folder = "/Workspace/Users/user/user/"

data_labels_filtered4 = pd.read_csv(output_folder + "data_labels_filtered4.csv")

matching_ids = set(data_labels_filtered4['Record ID #:']).intersection(set(subject_ids_to_check))

print(f"Total rows in data_labels_filtered4: {len(data_labels_filtered4)}")
print(f"Subject IDs in subject_ids_to_check: {len(subject_ids_to_check)}")
print(f"Matching subject IDs found: {len(matching_ids)}")

if matching_ids:
    print(matching_ids)
    display(data_labels_filtered4[data_labels_filtered4['Record ID #:'].isin(matching_ids)])
else:
    print("No matching subject IDs found.")

In [0]:
# Pull the 10 shortest unique notes from notes_enrolled4, restricted to subjects that also appear in results_GPT_full_df

valid_subjects = results_GPT_full_df['Subject ID'].unique()

filtered_notes = notes_enrolled4[
    notes_enrolled4['Subject ID'].isin(valid_subjects)
].dropna(subset=['NOTE_TEXT']).drop_duplicates(subset=['NOTE_TEXT']).copy()

filtered_notes['char_count'] = filtered_notes['NOTE_TEXT'].str.len()

smallest_10 = filtered_notes.nsmallest(10, 'char_count')[['Subject ID', 'NOTE_TEXT', 'char_count']]
display(smallest_10)

In [0]:
manual_score = manual_score[~manual_score['Subject ID'].isin([5 WES IDs])]
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin([5 WES IDs])]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin([5 WES IDs])]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin([5 WES IDs])]

In [0]:
# Drop rows from manual_score that don't have a matching Subject ID in results_GPT_full_df

gpt_ids = set(results_GPT_full_df['Subject ID'].unique())

before_count = len(manual_score)
manual_score = manual_score[manual_score['Subject ID'].isin(gpt_ids)].reset_index(drop=True)
after_count = len(manual_score)

print(f"Rows before: {before_count}")
print(f"Rows after: {after_count}")
print(f"Rows dropped: {before_count - after_count}")

In [0]:
# Find all notes (in notes_enrolled4) belonging to subjects present in any of the
# three model result sets, where the note text contains three consecutive asterisks ("***")

all_model_subject_ids = (
    set(results_GPT_full_df['Subject ID'])
    | set(results_mini_full_df['Subject ID'])
    | set(results_claude_full_df['Subject ID'])
)

notes_subset = notes_enrolled4[
    notes_enrolled4['Subject ID'].isin(all_model_subject_ids)
].dropna(subset=['NOTE_TEXT']).copy()

triple_asterisk_notes = notes_subset[
    notes_subset['NOTE_TEXT'].str.contains(r'\*\*\*', regex=True, na=False)
][['Subject ID', 'NOTE_TEXT']].reset_index(drop=True)

print(f"Notes containing '***': {len(triple_asterisk_notes)}")

if triple_asterisk_notes.empty:
    print("No matching notes found.")
else:
    display(triple_asterisk_notes)

In [0]:
# Viewing score totals across entire dataframes (no subsetting)

score_totals = {}

for name, df, score_col, agg in [
    ('Manual', manual_score, 'Phenotype', 'count'),
    ('GPT', results_GPT_full_df, 'GPT_Score', 'sum'),
    ('GPT Mini', results_mini_full_df, 'GPT_Mini_Score', 'sum'),
    ('Claude Sonnet', results_claude_full_df, 'Claude_Sonnet_Score', 'sum'),
]:
    if agg == 'count':
        score_totals[name] = df[score_col].count()
    else:
        score_totals[name] = df[score_col].sum()

print("Total phenotypes scored positive, per model, across all subjects:")
for model, total in score_totals.items():
    print(f"  {model}: {total}")

In [0]:
print(f"notes_enrolled4 subjects: {notes_enrolled4['Subject ID'].nunique()}")
print(f"results_GPT_full_df subjects: {results_GPT_full_df['Subject ID'].nunique()}")
print(f"results_mini_full_df subjects: {results_mini_full_df['Subject ID'].nunique()}")
print(f"results_claude_full_df subjects: {results_claude_full_df['Subject ID'].nunique()}")
print(f"manual_score subjects: {manual_score['Subject ID'].nunique()}")

In [0]:
# Create sets of (Subject ID, Phenotype) tuples for each model
gpt_pairs = set(zip(results_GPT_full_df['Subject ID'], results_GPT_full_df['Phenotype']))
mini_pairs = set(zip(results_mini_full_df['Subject ID'], results_mini_full_df['Phenotype']))
claude_pairs = set(zip(results_claude_full_df['Subject ID'], results_claude_full_df['Phenotype']))

# Check if all three are identical
print(f"All three models have identical Subject ID/Phenotype pairs: {gpt_pairs == mini_pairs == claude_pairs}")

# If not, see what's different
print("In GPT but not Mini:", gpt_pairs - mini_pairs)
print("In Mini but not GPT:", mini_pairs - gpt_pairs)
print("In GPT but not Claude:", gpt_pairs - claude_pairs)
print("In Claude but not GPT:", claude_pairs - gpt_pairs)

In [0]:
# Finding parse error details
parse_error_ids = results_claude_full_df[results_claude_full_df['Phenotype'] == 'PARSE ERROR']['Subject ID']
print(len(parse_error_ids))
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin(parse_error_ids)]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin(parse_error_ids)]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin(parse_error_ids)]
manual_score = manual_score[~manual_score['Subject ID'].isin(parse_error_ids)]

In [0]:
# Identify Subject IDs with Enrollment Status == 'Cancelled WES Testing' in notes_enrolled4,
# then drop those Subject IDs from notes_enrolled4, manual_score, and the three model result dataframes

cancelled_ids = set(
    notes_enrolled4.loc[notes_enrolled4['Enrollment Status'] == 'Cancelled WES Testing', 'Subject ID']
)

print(f"Subject IDs with Enrollment Status = 'Cancelled WES Testing': {len(cancelled_ids)}")
print(cancelled_ids)

before_counts = {
    'notes_enrolled4': notes_enrolled4['Subject ID'].nunique(),
    'manual_score': manual_score['Subject ID'].nunique(),
    'results_GPT_full_df': results_GPT_full_df['Subject ID'].nunique(),
    'results_mini_full_df': results_mini_full_df['Subject ID'].nunique(),
    'results_claude_full_df': results_claude_full_df['Subject ID'].nunique(),
}

notes_enrolled4 = notes_enrolled4[~notes_enrolled4['Subject ID'].isin(cancelled_ids)]
manual_score = manual_score[~manual_score['Subject ID'].isin(cancelled_ids)]
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin(cancelled_ids)]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin(cancelled_ids)]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin(cancelled_ids)]

after_counts = {
    'notes_enrolled4': notes_enrolled4['Subject ID'].nunique(),
    'manual_score': manual_score['Subject ID'].nunique(),
    'results_GPT_full_df': results_GPT_full_df['Subject ID'].nunique(),
    'results_mini_full_df': results_mini_full_df['Subject ID'].nunique(),
    'results_claude_full_df': results_claude_full_df['Subject ID'].nunique(),
}

print("\nUnique Subject IDs before -> after:")
for k in before_counts:
    print(f"  {k}: {before_counts[k]} -> {after_counts[k]} (dropped {before_counts[k] - after_counts[k]})")

In [0]:
print(f"results_GPT_full_df subjects: {results_GPT_full_df['Subject ID'].nunique()}")
print(f"results_mini_full_df subjects: {results_mini_full_df['Subject ID'].nunique()}")
print(f"results_claude_full_df subjects: {results_claude_full_df['Subject ID'].nunique()}")
print(f"manual_score subjects: {manual_score['Subject ID'].nunique()}")

In [0]:
# Phenotypes in manual_score but not GPT results
manual_score_not_in_GPT = manual_score[~manual_score['Phenotype'].isin(results_GPT_full_df['Phenotype'])]
print(manual_score_not_in_GPT)

In [0]:
# Removing phenotypes in manual_score that aren't in the GPT results. 
manual_score = manual_score[manual_score['Phenotype'].isin(results_GPT_full_df['Phenotype'])]

In [0]:
# Seeing which Subject IDs are in each AI but not the manual score
print("In GPT but not manual_score:", set(results_GPT_full_df['Subject ID']) - set(manual_score['Subject ID']))
print("In Mini but not manual_score:", set(results_mini_full_df['Subject ID']) - set(manual_score['Subject ID']))
print("In Claude but not manual_score:", set(results_claude_full_df['Subject ID']) - set(manual_score['Subject ID']))

In [0]:
# Subject IDs in notes_enrolled4 but not in manuaL_score
in_notes_not_manual = set(notes_enrolled4['Subject ID']) - set(manual_score['Subject ID'])
print(f"Subject IDs in notes_enrolled4 but not manual_score: {len(in_notes_not_manual)}")
print(in_notes_not_manual)

In [0]:
# Subject IDs in results_GPT_full_df but not in manual_score
gpt_not_manual_ids = set(results_GPT_full_df['Subject ID']) - set(manual_score['Subject ID'])

subset = notes_enrolled4[notes_enrolled4['Subject ID'].isin(gpt_not_manual_ids)]

if subset.empty:
    print("No subjects found in results_GPT_full_df that are missing from manual_score.")
else:
    display(subset)

In [0]:
#Combining all AI scoring to one dataframe

combined_df = results_GPT_full_df.merge(
    results_mini_full_df[['Subject ID', 'Phenotype', 'GPT_Mini_Score']],
    on=['Subject ID', 'Phenotype']
).merge(
    results_claude_full_df[['Subject ID', 'Phenotype', 'Claude_Sonnet_Score']],
    on=['Subject ID', 'Phenotype']
)

display(combined_df)

In [0]:
# Normalizing manual_score to match format of combined_df/AI scoring

# Get all unique subject IDs and phenotypes
all_subject_ids = combined_df['Subject ID'].unique()
all_phenotypes = checklist_filtered['Term from checklist'].dropna().unique()

# Create a complete grid of every (Subject ID, Phenotype) combination
full_grid = pd.MultiIndex.from_product(
    [all_subject_ids, all_phenotypes], 
    names=['Subject ID', 'Phenotype']
).to_frame(index=False)

# Mark which combinations were manually scored as present
manual_score['Manual_Score'] = 1
full_grid = full_grid.merge(
    manual_score[['Subject ID', 'Phenotype', 'Manual_Score']], 
    on=['Subject ID', 'Phenotype'], 
    how='left'
)
full_grid['Manual_Score'] = full_grid['Manual_Score'].fillna(0).astype(int)

display(full_grid)
print(len(full_grid))

In [0]:
# Quality auditing the normalized manual score output

print("Unique Subject IDs in all_subject_ids:", len(all_subject_ids))
print("Unique Phenotypes in all_phenotypes:", len(all_phenotypes))
print(f"Expected grid size: {len(all_subject_ids)} x {len(all_phenotypes)} = {len(all_subject_ids) * len(all_phenotypes)}")

# Check for duplicates in manual_score after the merge
print("\nDuplicate (Subject ID, Phenotype) pairs in full_grid:")
print(full_grid.duplicated(subset=['Subject ID', 'Phenotype']).sum())

# Check if any phenotypes in manual_score aren't in checklist_filtered
manual_phens = set(manual_score['Phenotype'])
checklist_phens = set(checklist_filtered['Term from checklist'])
print("\nPhenotypes in manual_score but not in checklist_filtered:")
print(manual_phens - checklist_phens)

In [0]:
# Troubleshooting 126 phenotypes. Possibly null values?
print("Null phenotypes in manual_score:", manual_score['Phenotype'].isna().sum())
print("Null phenotypes in checklist_filtered:", checklist_filtered['Term from checklist'].isna().sum())

In [0]:
# Dropping null phenotype values from checklist_filtered
checklist_filtered = checklist_filtered.dropna(subset=['Term from checklist'])
print(f"Checklist phenotypes after dropping nulls: {len(checklist_filtered)}")

In [0]:
# Showing and dropping duplicate phenotypes
# Show the duplicate terms
print("Duplicate terms:")
print(checklist_filtered[checklist_filtered['Term from checklist'].duplicated(keep=False)]['Term from checklist'].unique())

# Deduplicate
checklist_filtered = checklist_filtered.drop_duplicates(subset='Term from checklist')
print(f"\nChecklist phenotypes after deduplication: {len(checklist_filtered)}")

In [0]:
# Find the duplicate phenotype in checklist_filtered
print("Duplicate phenotypes in checklist_filtered:")
print(checklist_filtered[checklist_filtered['Term from checklist'].duplicated(keep=False)]['Term from checklist'].unique())

# Find duplicate (Subject ID, Phenotype) pairs in manual_score
print("\nDuplicate (Subject ID, Phenotype) pairs in manual_score:")
print(manual_score[manual_score.duplicated(subset=['Subject ID', 'Phenotype'], keep=False)])

## Building combined_df

In [0]:
#Rebuilding manual_score with clean checklist_filtered

# Recreate records from REDCap_mapping
records = []
for i in range(0, len(REDCap_mapping)):
    phen = []
    for j in range(0, len(REDCap_mapping.columns)):
        if pd.isnull(REDCap_mapping.iloc[i][j]) == False:
            phen.append(REDCap_mapping.iloc[i][j])
    records.append(phen)

records_formatted = pd.DataFrame(records)

# Recreate manual_list from records_formatted
manual_list = []
for j in range(0, len(records_formatted)):
    for i in range(0, len(records_formatted.columns)):
        manual_list.append(records_formatted[i][j])

manual_list = [x for x in manual_list if x is not None]

# Rebuild manual_score with clean checklist_filtered
rows = []
current_id = None
for item in manual_list:
    if item.startswith('WES'):
        current_id = item
    else:
        rows.append((current_id, item))

manual_df = pd.DataFrame(rows, columns=['Subject ID', 'Phenotype'])

manual_score = manual_df.merge(
    checklist_filtered[['Term from checklist', 'HPO term (ID)']],
    left_on='Phenotype',
    right_on='Term from checklist',
    how='inner'
).drop(columns=['Term from checklist'])

manual_score['Phenotype'] = manual_score['Phenotype'].str.strip()
print(f"manual_score rows: {len(manual_score)}")
print(f"Duplicate (Subject ID, Phenotype) pairs: {manual_score.duplicated(subset=['Subject ID', 'Phenotype']).sum()}")
display(manual_score)

In [0]:
# Normalizing the manual_score so it can be joined onto the combined dataframe 
# Get all unique subject IDs and phenotypes
all_subject_ids = combined_df['Subject ID'].unique()
all_phenotypes = checklist_filtered['Term from checklist'].unique()

# Create a complete grid of every (Subject ID, Phenotype) combination
full_grid = pd.MultiIndex.from_product(
    [all_subject_ids, all_phenotypes], 
    names=['Subject ID', 'Phenotype']
).to_frame(index=False)

# Mark which combinations were manually scored as present
manual_score['Manual_Score'] = 1
full_grid = full_grid.merge(
    manual_score[['Subject ID', 'Phenotype', 'Manual_Score']], 
    on=['Subject ID', 'Phenotype'], 
    how='left'
)
full_grid['Manual_Score'] = full_grid['Manual_Score'].fillna(0).astype(int)

print(len(full_grid))
print("Duplicate (Subject ID, Phenotype) pairs:", full_grid.duplicated(subset=['Subject ID', 'Phenotype']).sum())
display(full_grid)

In [0]:
# Adding the manual score values to the AI combined dataframe from earlier
combined_df = combined_df.merge(full_grid, on=['Subject ID', 'Phenotype'], how='left')
print(f"combined_df rows: {len(combined_df)}")
print(f"Null Manual_Score values: {combined_df['Manual_Score'].isna().sum()}")
display(combined_df)

## Disagreements With Manual

In [0]:
# Flag disagreements for each model vs manual
combined_df['GPT_disagree'] = (combined_df['GPT_Score'] != combined_df['Manual_Score']).astype(int)
combined_df['GPT_Mini_disagree'] = (combined_df['GPT_Mini_Score'] != combined_df['Manual_Score']).astype(int)
combined_df['Claude_Sonnet_disagree'] = (combined_df['Claude_Sonnet_Score'] != combined_df['Manual_Score']).astype(int)

# Total disagreements — any model disagrees with manual
combined_df['any_disagree'] = (
    (combined_df['GPT_disagree'] == 1) | 
    (combined_df['GPT_Mini_disagree'] == 1) | 
    (combined_df['Claude_Sonnet_disagree'] == 1)
).astype(int)

print(f"Total rows where at least one model disagrees with manual: {combined_df['any_disagree'].sum()}")

# Unique disagreements — only this model disagrees with manual, other two agree
combined_df['GPT_unique_disagree'] = (
    (combined_df['GPT_disagree'] == 1) & 
    (combined_df['GPT_Mini_disagree'] == 0) & 
    (combined_df['Claude_Sonnet_disagree'] == 0)
).astype(int)

combined_df['GPT_Mini_unique_disagree'] = (
    (combined_df['GPT_Mini_disagree'] == 1) & 
    (combined_df['GPT_disagree'] == 0) & 
    (combined_df['Claude_Sonnet_disagree'] == 0)
).astype(int)

combined_df['Claude_Sonnet_unique_disagree'] = (
    (combined_df['Claude_Sonnet_disagree'] == 1) & 
    (combined_df['GPT_disagree'] == 0) & 
    (combined_df['GPT_Mini_disagree'] == 0)
).astype(int)

print("\nTotal disagreements vs manual per model:")
print(f"  GPT: {combined_df['GPT_disagree'].sum()}")
print(f"  GPT Mini: {combined_df['GPT_Mini_disagree'].sum()}")
print(f"  Claude Sonnet: {combined_df['Claude_Sonnet_disagree'].sum()}")

print("\nUnique disagreements vs manual (only this model disagrees):")
print(f"  GPT: {combined_df['GPT_unique_disagree'].sum()}")
print(f"  GPT Mini: {combined_df['GPT_Mini_unique_disagree'].sum()}")
print(f"  Claude Sonnet: {combined_df['Claude_Sonnet_unique_disagree'].sum()}")

In [0]:
print("Unique values in Claude_Sonnet_Score:", combined_df['Claude_Sonnet_Score'].unique())
print("Unique values in GPT_Score:", combined_df['GPT_Score'].unique())
print("Unique values in GPT_Mini_Score:", combined_df['GPT_Mini_Score'].unique())
print("Unique values in Manual_Score:", combined_df['Manual_Score'].unique())

In [0]:
# GPT agrees but Mini disagrees
gpt_agree_mini_disagree = (
    (combined_df['GPT_disagree'] == 0) & 
    (combined_df['GPT_Mini_disagree'] == 1)
).sum()

# GPT agrees but Claude disagrees
gpt_agree_claude_disagree = (
    (combined_df['GPT_disagree'] == 0) & 
    (combined_df['Claude_Sonnet_disagree'] == 1)
).sum()

print(f"GPT agrees, Mini disagrees: {gpt_agree_mini_disagree}")
print(f"GPT agrees, Claude disagrees: {gpt_agree_claude_disagree}")

In [0]:
import matplotlib
print(matplotlib.__version__)
print(matplotlib.__file__)

In [0]:
# Number of subjects with a disagreement
disagree_by_subject = combined_df.groupby('Subject ID').agg(
    any_disagree=('any_disagree', 'sum')
).reset_index()

subjects_with_disagree = (disagree_by_subject['any_disagree'] > 0).sum()
subjects_no_disagree = (disagree_by_subject['any_disagree'] == 0).sum()

print(f"Total subjects: {len(disagree_by_subject)}")
print(f"Subjects with at least one disagreement: {subjects_with_disagree}")
print(f"Subjects with no disagreements: {subjects_no_disagree}")
print(f"Percent of subjects with at least one disagreement: {subjects_with_disagree / len(disagree_by_subject) * 100:.1f}%")

In [0]:
# Check for duplicate Subject IDs in notes_enrolled4
dup_check = notes_enrolled4['Subject ID'].duplicated().sum()
print(f"Total rows in notes_enrolled4: {len(notes_enrolled4)}")
print(f"Unique Subject IDs: {notes_enrolled4['Subject ID'].nunique()}")
print(f"Duplicate Subject ID rows: {dup_check}")

In [0]:
combined_df_full_info = combined_df.merge(
    notes_enrolled4,
    on='Subject ID',
    how='left'
)

print(f"combined_df rows: {len(combined_df)}")
print(f"combined_df_full_info rows: {len(combined_df_full_info)}")
print(f"Row count matches: {len(combined_df) == len(combined_df_full_info)}")
print(f"Null values introduced by merge (any column): {combined_df_full_info.isna().sum().sum() - combined_df.isna().sum().sum()}")

display(combined_df_full_info)

In [0]:
print(combined_df_full_info.shape)
print(combined_df_full_info['NOTE_TEXT'].duplicated().sum())
print(combined_df_full_info['NOTE_TEXT'].nunique())

In [0]:
# Pull the 10 unique notes with the smallest character counts, including Subject ID
unique_notes_df = combined_df_full_info.dropna(subset=['NOTE_TEXT']).drop_duplicates(subset=['NOTE_TEXT']).copy()
unique_notes_df['char_count'] = unique_notes_df['NOTE_TEXT'].str.len()

smallest_10 = unique_notes_df.nsmallest(10, 'char_count')[['Subject ID', 'NOTE_TEXT', 'char_count']]
display(smallest_10)

In [0]:
# How many rows per subject, and how many are exact duplicates?
rows_per_subject = combined_df_full_info.groupby('Subject ID').size()
print(rows_per_subject.describe())

# If you have a phenotype/category column, compare row count vs unique phenotype count per subject
if 'Phenotype' in combined_df_full_info.columns:
    check = combined_df_full_info.groupby('Subject ID').agg(
        total_rows=('Phenotype', 'size'),
        unique_phenotypes=('Phenotype', 'nunique')
    )
    print(check[check['total_rows'] != check['unique_phenotypes']])

In [0]:
# Confirm each subject has exactly one distinct NOTE_TEXT (and one char_count) across their 125 rows
notes_per_subject = combined_df_full_info.groupby('Subject ID')['NOTE_TEXT'].nunique()
print(notes_per_subject.value_counts())

In [0]:
categorical_cols = ['CURRENT_AUTHOR', 'AUTHOR_ROLE', 'UPDATER', 'COSIGNER']

for col in categorical_cols:
    n_unique = combined_df_full_info[col].nunique(dropna=True)
    n_missing = combined_df_full_info[col].isna().sum()
    print(f"{col}: {n_unique} unique values, {n_missing} missing rows")
    print(combined_df_full_info[col].value_counts(dropna=True).head(15))
    print()

In [0]:
n_unique = notes_enrolled4['COSIGNER'].nunique(dropna=True)
n_missing = notes_enrolled4['COSIGNER'].isna().sum()

print(f"COSIGNER: {n_unique} unique values, {n_missing} missing rows")
print(notes_enrolled4['COSIGNER'].value_counts(dropna=True))

In [0]:
!pip install openpyxl pandas



file_path = "/Workspace/Users/user/user/UMD Enrollment Log_11.7.2025_for_Gabriel 2.xlsx"

umd_enrollment_log = pd.read_excel(file_path)
print(umd_enrollment_log.columns)

In [0]:
# Quality check: confirm each Subject ID maps to exactly one unique
# 'Consented By:' value after the merge (i.e. no subject has conflicting values)

consented_per_subject = umd_enrollment_log.dropna(subset=['Consented By:']).groupby('Subject ID')['Consented By:'].nunique()

n_multi = (consented_per_subject > 1).sum()
print(f"Subjects with more than one distinct 'Consented By:' value: {n_multi}")

if n_multi > 0:
    problem_subjects = consented_per_subject[consented_per_subject > 1].index
    print("Subjects with conflicting values:")
    display(
        umd_enrollment_log[umd_enrollment_log['Subject ID'].isin(problem_subjects)][['Subject ID', 'Consented By:']]
        .drop_duplicates()
        .sort_values('Subject ID')
    )
else:
    print("Every subject has exactly one unique 'Consented By:' value. Merge looks clean.")

# Subjects with no matching 'Consented By:' value after the merge (i.e. didn't match
# to anything in umd_enrollment_log, or matched but the field itself was blank)

subjects_no_consent = umd_enrollment_log[umd_enrollment_log['Consented By:'].isna()][['Subject ID']].drop_duplicates().sort_values('Subject ID')

print(f"Subjects with no 'Consented By:' value: {len(subjects_no_consent)}")
display(subjects_no_consent)

In [0]:
# Count of subjects where Manual scored 'Autism Spectrum Disorder (ASD)' as positive

asd_manual_count = df[
    (df['Phenotype'] == 'Autism Spectrum Disorder (ASD)') &
    (df['Manual_Score'] == 1)
]['Subject ID'].nunique()

total_asd_rows = (df['Phenotype'] == 'Autism Spectrum Disorder (ASD)').sum()

print(f"Subjects with 'Autism Spectrum Disorder (ASD)' scored positive by Manual: {asd_manual_count}")
print(f"Total subjects with an ASD phenotype row: {total_asd_rows}")

In [0]:
import requests
import json
import os

output_folder = "/Workspace/Users/user/user/"
hp_json_path = output_folder + "hp.json"

# One-time download of the full HPO ontology (official OBO Foundry release).
# Only run this once -- after this, everything is done locally from the saved file.
if not os.path.exists(hp_json_path):
    url = "http://purl.obolibrary.org/obo/hp.json"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    with open(hp_json_path, 'wb') as f:
        f.write(resp.content)
    print(f"Downloaded HPO ontology to {hp_json_path} ({len(resp.content)/1e6:.1f} MB)")
else:
    print(f"HPO ontology already downloaded at {hp_json_path}, skipping.")

In [0]:
# ------------------------------------------------------------
# Parse the local hp.json file -- no network calls below this point
# ------------------------------------------------------------
with open(hp_json_path, 'r') as f:
    hpo_data = json.load(f)

graph = hpo_data['graphs'][0]

def iri_to_short(iri):
    # e.g. "http://purl.obolibrary.org/obo/HP_0001250" -> "HP:0001250"
    frag = iri.rsplit('/', 1)[-1]
    return frag.replace('_', ':', 1)

def short_to_iri(short_id):
    # e.g. "HP:0001250" -> "http://purl.obolibrary.org/obo/HP_0001250"
    return "http://purl.obolibrary.org/obo/" + short_id.replace(':', '_', 1)

# id (short form) -> label
id_to_label = {
    iri_to_short(n['id']): n.get('lbl', '')
    for n in graph['nodes'] if 'lbl' in n
}

# child (short form) -> set of parent ids (short form), from is_a edges only
parent_map = {}
for e in graph['edges']:
    if e.get('pred') in ('is_a', 'http://www.w3.org/2000/01/rdf-schema#subClassOf'):
        child = iri_to_short(e['sub'])
        parent = iri_to_short(e['obj'])
        parent_map.setdefault(child, set()).add(parent)

# Top-level categories = direct children of HP:0000118 (Phenotypic abnormality)
top_level_ids = {
    child for child, parents in parent_map.items() if 'HP:0000118' in parents
}
print(f"Top-level HPO categories found: {len(top_level_ids)}")
for tid in sorted(top_level_ids):
    print(f"  {tid}: {id_to_label.get(tid)}")

def find_top_level_category(hpo_id, parent_map, top_level_ids, id_to_label):
    """BFS up the parent hierarchy until a top-level category is hit."""
    if hpo_id in top_level_ids:
        return id_to_label.get(hpo_id, hpo_id)
    visited = set()
    frontier = [hpo_id]
    while frontier:
        next_frontier = []
        for node in frontier:
            if node in visited:
                continue
            visited.add(node)
            for parent in parent_map.get(node, []):
                if parent in top_level_ids:
                    return id_to_label.get(parent, parent)
                next_frontier.append(parent)
        frontier = next_frontier
    return 'Unmapped / Other'

# Build the HPO ID -> category lookup for the phenotypes in the checklist
phenotype_to_hpo = (
    checklist_filtered[['Term from checklist', 'HPO term (ID)']]
    .dropna()
    .drop_duplicates(subset='Term from checklist')
    .set_index('Term from checklist')['HPO term (ID)']
    .to_dict()
)

hpo_to_category = {
    hpo_id: find_top_level_category(hpo_id, parent_map, top_level_ids, id_to_label)
    for hpo_id in set(phenotype_to_hpo.values())
}

print(f"\nMapped {len(hpo_to_category)} unique HPO IDs to categories.")
display(pd.DataFrame(hpo_to_category.items(), columns=['HPO_ID', 'HPO_Category']))

In [0]:
def get_all_top_level_categories(hpo_id, parent_map, top_level_ids):
    visited = set()
    found = set()
    frontier = [hpo_id]
    while frontier:
        next_frontier = []
        for node in frontier:
            if node in visited:
                continue
            visited.add(node)
            if node in top_level_ids:
                found.add(node)
                continue  # don't traverse past a top-level hit
            for parent in parent_map.get(node, []):
                next_frontier.append(parent)
        frontier = next_frontier
    return found

multi_parent_report = {
    hpo_id: get_all_top_level_categories(hpo_id, parent_map, top_level_ids)
    for hpo_id in set(phenotype_to_hpo.values())
}
multi_parent_terms = {k: v for k, v in multi_parent_report.items() if len(v) > 1}
print(f"Phenotypes with more than one top-level HPO category: {len(multi_parent_terms)}")
for hpo_id, cats in multi_parent_terms.items():
    print(f"  {hpo_id}: {[id_to_label.get(c, c) for c in cats]}")

In [0]:
# ============================================================
# Check ontology depth (distance from HP:0000118) for every phenotype
# ============================================================

def compute_depth(hpo_id, parent_map, root='HP:0000118'):
    """Shortest number of is_a steps from hpo_id up to the root (HP:0000118).
    Returns None if no path to root is found."""
    if hpo_id == root:
        return 0
    from collections import deque
    visited = {hpo_id}
    queue = deque([(hpo_id, 0)])
    while queue:
        node, dist = queue.popleft()
        for parent in parent_map.get(node, []):
            if parent == root:
                return dist + 1
            if parent not in visited:
                visited.add(parent)
                queue.append((parent, dist + 1))
    return None  # no path found -- possible data issue, worth flagging

depth_records = []
for pheno, hpo_id in phenotype_to_hpo.items():
    depth = compute_depth(hpo_id, parent_map)
    depth_records.append({
        'Phenotype': pheno,
        'HPO_ID': hpo_id,
        'HPO_Label': id_to_label.get(hpo_id, ''),
        'Depth': depth,
    })

depth_df = pd.DataFrame(depth_records).sort_values('Depth')

print(f"Total phenotypes checked: {len(depth_df)}")
print(f"Unique depth values found: {sorted(depth_df['Depth'].dropna().unique())}")
print(f"\nDepth distribution:")
print(depth_df['Depth'].value_counts().sort_index())

no_path = depth_df[depth_df['Depth'].isna()]
if len(no_path) > 0:
    print(f"\nWARNING: {len(no_path)} phenotype(s) had no traceable path to HP:0000118 -- check these IDs:")
    display(no_path)

display(depth_df)

In [0]:
def get_ancestor_at_depth(hpo_id, parent_map, target_depth, root='HP:0000118'):
    """Find the ancestor of hpo_id that sits at target_depth steps from root,
    along the shortest path. Returns None if hpo_id's depth is already <= target_depth
    with no ancestor exactly at that depth."""
    from collections import deque
    depth = compute_depth(hpo_id, parent_map, root)
    if depth is None or depth <= target_depth:
        return hpo_id  # already at or above the target depth -- keep as-is
    # BFS tracking path length, return first ancestor hit at target_depth
    visited = {hpo_id}
    queue = deque([(hpo_id, depth)])
    while queue:
        node, d = queue.popleft()
        if d == target_depth:
            return node
        for parent in parent_map.get(node, []):
            if parent not in visited:
                visited.add(parent)
                parent_depth = compute_depth(parent, parent_map, root)
                if parent_depth is not None:
                    queue.append((parent, parent_depth))
    return None

depth_df['Depth2_Ancestor'] = depth_df['HPO_ID'].apply(
    lambda h: get_ancestor_at_depth(h, parent_map, target_depth=2)
)
depth_df['Depth2_Label'] = depth_df['Depth2_Ancestor'].map(id_to_label)

n_original = depth_df['Phenotype'].nunique()
n_depth2_buckets = depth_df['Depth2_Ancestor'].nunique()
n_top_level = len(top_level_ids)

print(f"Original distinct phenotypes: {n_original}")
print(f"Distinct depth-2 buckets: {n_depth2_buckets}")
print(f"Distinct top-level (depth-1) categories: {n_top_level}")
print(f"\nHow many original phenotypes collapse into each depth-2 bucket:")
display(depth_df.groupby('Depth2_Label')['Phenotype'].apply(list))

In [0]:
seven_imputed_ids = [7 WES IDs]
set(seven_imputed_ids) & set(results_GPT_full_df['Subject ID'])

# Adjudication Splitting

In [0]:
# Randomly splitting the data into adjudication groups

import numpy as np

# Get per-subject disagreement stats
subject_stats = combined_df.groupby('Subject ID').agg(
    total_phenotypes=('Manual_Score', 'sum'),
    any_disagree=('any_disagree', 'sum')
).reset_index()

print(f"Total subjects: {len(subject_stats)}")
print(f"Mean phenotypes per subject: {subject_stats['total_phenotypes'].mean():.2f}")
print(f"Std phenotypes per subject: {subject_stats['total_phenotypes'].std():.2f}")

# Shuffle randomly
subject_stats = subject_stats.sample(frac=1, random_state=42).reset_index(drop=True)

# Split into 4 groups: 1 group of 50, 3 equal groups of remaining
n_total = len(subject_stats)
n_remaining = n_total - 50
n_per_group = n_remaining // 3
n_extra = n_remaining % 3  # distribute any remainder

print(f"\nGroup sizes: 50 + {n_per_group} + {n_per_group} + {n_per_group + n_extra}")

# Sort by phenotype count for stratified-style assignment
# Assign groups by cycling through sorted subjects so each group
# gets an even spread of high and low phenotype counts
subject_stats_sorted = subject_stats.sort_values('total_phenotypes').reset_index(drop=True)

# Assign group labels cycling 0-3 across sorted subjects
subject_stats_sorted['group'] = subject_stats_sorted.index % 4

# Remap group 0 to the "50 note" group and cap it at 50,
# redistributing overflow into groups 1-3
group0 = subject_stats_sorted[subject_stats_sorted['group'] == 0].copy()
overflow = group0.iloc[50:]
group0 = group0.iloc[:50]

# Redistribute overflow evenly into groups 1-3
for i, idx in enumerate(overflow.index):
    subject_stats_sorted.loc[idx, 'group'] = (i % 3) + 1

subject_stats_sorted.loc[group0.index, 'group'] = 0

# Report group stats
print("\n--- Group summary ---")
for g in range(4):
    grp = subject_stats_sorted[subject_stats_sorted['group'] == g]
    label = "Group 0 (n=50)" if g == 0 else f"Group {g}"
    print(f"{label}: n={len(grp)}, mean phenotypes={grp['total_phenotypes'].mean():.2f}, std={grp['total_phenotypes'].std():.2f}")

# Save group assignments
group_assignments = subject_stats_sorted[['Subject ID', 'group']]
display(group_assignments)

In [0]:
print("\n--- Group summary ---")
for g in range(4):
    grp = subject_stats_sorted[subject_stats_sorted['group'] == g]
    label = "Group 0 (n=50)" if g == 0 else f"Group {g}"
    n_notes = len(grp)
    total_pheno = grp['total_phenotypes'].sum()
    avg_pheno_per_note = grp['total_phenotypes'].mean()
    print(f"{label}:")
    print(f"  Notes:                    {n_notes}")
    print(f"  Total phenotypes:         {total_pheno}")
    print(f"  Avg phenotypes per note:  {avg_pheno_per_note:.2f}")
    print(f"  Std phenotypes per note:  {grp['total_phenotypes'].std():.2f}")
    print()

In [0]:
# Recompute subject_stats using disagreements 
subject_stats = combined_df.groupby('Subject ID').agg(
    total_disagreements=('any_disagree', 'sum')
).reset_index()

print(f"Total subjects: {len(subject_stats)}")
print(f"Total disagreements: {subject_stats['total_disagreements'].sum()}")
print(f"Mean disagreements per subject: {subject_stats['total_disagreements'].mean():.2f}")

results = []

for seed in range(1000):
    shuffled = subject_stats.sample(frac=1, random_state=seed).reset_index(drop=True)
    g0 = shuffled.iloc[:50]
    remaining = shuffled.iloc[50:].reset_index(drop=True)
    n_per_group = len(remaining) // 3
    n_extra = len(remaining) % 3
    group_sizes = [n_per_group + (1 if i < n_extra else 0) for i in range(3)]

    start = 0
    means = [g0['total_disagreements'].mean()]
    for size in group_sizes:
        grp = remaining.iloc[start:start+size]
        means.append(grp['total_disagreements'].mean())
        start += size

    results.append({'seed': seed, 'std_of_means': np.std(means), 'means': means})

results_df = pd.DataFrame(results).sort_values('std_of_means')
print("\nTop 10 most balanced seeds:")
display(results_df.head(10))

best_seed = results_df.iloc[0]['seed']
print(f"\nBest seed: {best_seed}")
print(f"Group means: {results_df.iloc[0]['means']}")

In [0]:
# Build the adjudication dataframes using the shuffled subject stats

subject_stats_shuffled = subject_stats.sample(frac=1, random_state=best_seed).reset_index(drop=True)

group0 = subject_stats_shuffled.iloc[:50].copy()
group0['group'] = 0

remaining = subject_stats_shuffled.iloc[50:].copy().reset_index(drop=True)
n_per_group = len(remaining) // 3
n_extra = len(remaining) % 3

group_sizes = [n_per_group + (1 if i < n_extra else 0) for i in range(3)]
print(f"Group sizes: {group_sizes}")

remaining['group'] = -1
start = 0
for i, size in enumerate(group_sizes):
    remaining.loc[start:start+size-1, 'group'] = i + 1
    start += size

subject_stats_sorted = pd.concat([group0, remaining], ignore_index=True)

print("--- Group summary ---")
for g in range(4):
    grp = subject_stats_sorted[subject_stats_sorted['group'] == g]
    label = "Group 0 (n=50)" if g == 0 else f"Group {g}"
    print(f"{label}:")
    print(f"  Notes:                      {len(grp)}")
    print(f"  Total disagreements:        {grp['total_disagreements'].sum()}")
    print(f"  Avg disagreements per note: {grp['total_disagreements'].mean():.2f}")
    print(f"  Std disagreements per note: {grp['total_disagreements'].std():.2f}")
    print()

print(f"Sum of all disagreements across groups: {subject_stats_sorted['total_disagreements'].sum()}")

group_assignments = subject_stats_sorted[['Subject ID', 'group']]
display(group_assignments)

In [0]:
print("--- subject_stats_sorted ---")
print(f"Total notes:         {subject_stats_sorted['Subject ID'].nunique()}")
print(f"Total disagreements: {subject_stats_sorted['total_disagreements'].sum()}")

print("\n--- combined_df ---")
print(f"Total notes:         {combined_df['Subject ID'].nunique()}")
print(f"Total disagreements: {combined_df['any_disagree'].sum()}")
print(f"Total rows:          {len(combined_df)}")

In [0]:
# Filter to only rows where any model disagrees with manual
adjudication_df = combined_df[combined_df['any_disagree'] == 1].copy()

# Join note text
adjudication_df = adjudication_df.merge(
    notes_enrolled4[['Subject ID', 'NOTE_TEXT']],
    on='Subject ID',
    how='left'
)

# Join group assignments
adjudication_df = adjudication_df.merge(
    group_assignments,
    on='Subject ID',
    how='left'
)

# Select and reorder columns
adjudication_df = adjudication_df[[
    'Subject ID',
    'Phenotype',
    'GPT_Score',
    'Justification',
    'Quote',
    'NOTE_TEXT',
    'Manual_Score',
    'GPT_Mini_Score',
    'Claude_Sonnet_Score',
    'group'
]]

print(f"Total rows: {len(adjudication_df)}")
print(f"Total subjects: {adjudication_df['Subject ID'].nunique()}")
print(f"Null NOTE_TEXT: {adjudication_df['NOTE_TEXT'].isna().sum()}")
print(f"Null group: {adjudication_df['group'].isna().sum()}")
display(adjudication_df.head())

In [0]:
# Checking that all dropped subjects are genuinely those where there are no disagreements on any of the phenotypes

no_disagree_subjects = combined_df.groupby('Subject ID')['any_disagree'].sum()
no_disagree_subjects = no_disagree_subjects[no_disagree_subjects == 0].index.tolist()
print(f"Subjects with no disagreements: {len(no_disagree_subjects)}")
print(f"Subject IDs: {no_disagree_subjects}")
print(f"Combined_df subjects: {combined_df['Subject ID'].nunique()}")
print(f"Adjudication_df subjects: {adjudication_df['Subject ID'].nunique()}")
print(f"Difference: {combined_df['Subject ID'].nunique() - adjudication_df['Subject ID'].nunique()}")

In [0]:
# Displaying the disagreements split as a table

summary_rows = []
for g in range(4):
    grp = adjudication_df[adjudication_df['group'] == g]
    label = "Group 0 (n=50)" if g == 0 else f"Group {g}"
    per_subject = grp.groupby('Subject ID').size()
    summary_rows.append({
        'Group': label,
        'Notes': grp['Subject ID'].nunique(),
        'Total Disagreements': len(grp),
        'Avg Disagreements per Note': round(per_subject.mean(), 2),
        'Std Disagreements per Note': round(per_subject.std(), 2)
    })

summary_table = pd.DataFrame(summary_rows)
print(summary_table.to_string(index=False))

In [0]:
print(adjudication_df.columns.tolist())

In [0]:
# Rebuild adjudication_df cleanly from combined_df
adjudication_df = combined_df[combined_df['any_disagree'] == 1].copy()

print(f"Rows with any disagreement: {len(adjudication_df)}")

# Join note text
adjudication_df = adjudication_df.merge(
    notes_enrolled4[['Subject ID', 'NOTE_TEXT']],
    on='Subject ID',
    how='left'
)
print(f"After NOTE_TEXT join: {len(adjudication_df)} rows, {adjudication_df['NOTE_TEXT'].isna().sum()} null NOTE_TEXT")

# Join group assignments
adjudication_df = adjudication_df.merge(
    group_assignments,
    on='Subject ID',
    how='left'
)
print(f"After group join: {len(adjudication_df)} rows, {adjudication_df['group'].isna().sum()} null group")

# Select and reorder columns
adjudication_df = adjudication_df[[
    'Subject ID',
    'Phenotype',
    'GPT_Score',
    'Justification',
    'Quote',
    'NOTE_TEXT',
    'Manual_Score',
    'GPT_Mini_Score',
    'Claude_Sonnet_Score',
    'group'
]]

# Split into four groups
group0_df = adjudication_df[adjudication_df['group'] == 0].copy()
group1_df = adjudication_df[adjudication_df['group'] == 1].copy()
group2_df = adjudication_df[adjudication_df['group'] == 2].copy()
group3_df = adjudication_df[adjudication_df['group'] == 3].copy()

print(f"\nGroup 0: {group0_df['Subject ID'].nunique()} subjects, {len(group0_df)} rows")
print(f"Group 1: {group1_df['Subject ID'].nunique()} subjects, {len(group1_df)} rows")
print(f"Group 2: {group2_df['Subject ID'].nunique()} subjects, {len(group2_df)} rows")
print(f"Group 3: {group3_df['Subject ID'].nunique()} subjects, {len(group3_df)} rows")
print(f"Total rows: {len(group0_df) + len(group1_df) + len(group2_df) + len(group3_df)}")
display(adjudication_df.head())

In [0]:
display(group0_df)
print(len(group0_df))
display(group1_df)
print(len(group1_df))
display(group2_df)
print(len(group2_df))
display(group3_df)
print(len(group3_df))

In [0]:
# Dropping unnecessary columns
cols = ['Subject ID', 'Phenotype', 'Justification', 'NOTE_TEXT', 'Quote']

group0_df = group0_df[cols].copy()
group1_df = group1_df[cols].copy()
group2_df = group2_df[cols].copy()
group3_df = group3_df[cols].copy()

display(group0_df.head())

In [0]:
group0_df.to_csv("/Workspace/Users/user/user/Kappa_Adjudications_Unscored.csv", index=False)
group1_df.to_csv("/Workspace/Users/user/user/Burak_Adjudications_Unscored.csv", index=False)
group2_df.to_csv("/Workspace/Users/user/user/Angie_Adjudications_Unscored.csv", index=False)
group3_df.to_csv("/Workspace/Users/user/user/Dustin_Adjudications_Unscored.csv", index=False)
print("Saved all four CSVs")

In [0]:
for name, df in [('Adjudication', group0_df), ('Burak', group1_df), ('Angie', group2_df), ('Dustin', group3_df)]:
    print(f"{name}:")
    print(f"  Rows: {len(df)}")
    print(f"  Unique subjects: {df['Subject ID'].nunique()}")
    print(f"  Phenotypes needing scoring: {len(df)}")
    print(f"  Estimated tokens (chars/4): {df['NOTE_TEXT'].str.len().sum() // 4:,}")
    print("---")

# Fixing Mistake

In [0]:
checklist_terms_now = checklist_filtered[['Term from checklist']].drop_duplicates()

pipeline_join = manual_df.merge(
    checklist_terms_now,
    left_on='Phenotype',
    right_on='Term from checklist',
    how='inner'
)

in_manual_score = set(zip(manual_score['Subject ID'], manual_score['Phenotype']))
pipeline_join['pair'] = list(zip(pipeline_join['Subject ID'], pipeline_join['Phenotype']))

pipeline_missing = pipeline_join[~pipeline_join['pair'].isin(in_manual_score)]

print(f"Rows in manual_df x current checklist_filtered: {len(pipeline_join)}")
print(f"Rows currently in manual_score: {len(manual_score)}")
print(f"Present per manual_df, absent from manual_score: {len(pipeline_missing)}")
print()
print(pipeline_missing.groupby('Phenotype').size().sort_values(ascending=False))

In [0]:
cohort_ids = set(combined_df['Subject ID'])

pipeline_missing_cohort = pipeline_missing[pipeline_missing['Subject ID'].isin(cohort_ids)]

print(f"Present per manual_df, absent from manual_score (unfiltered): {len(pipeline_missing)}")
print(f"Same, restricted to subjects in combined_df: {len(pipeline_missing_cohort)}")
print()
print(pipeline_missing_cohort.groupby('Phenotype').size().sort_values(ascending=False))

In [0]:
group0 = pd.read_csv("/Workspace/Users/user/user/Kappa_Adjudications_Unscored.csv")
group1 = pd.read_csv("/Workspace/Users/user/user/Burak_Adjudications_Unscored.csv")
group2 = pd.read_csv("/Workspace/Users/user/user/Angie_Adjudications_Unscored.csv")
group3 = pd.read_csv("/Workspace/Users/user/user/Dustin_Adjudications_Unscored.csv")

all_adjudications = pd.concat([group0, group1, group2, group3], ignore_index=True)

print(f"Rows loaded per file: {len(group0)}, {len(group1)}, {len(group2)}, {len(group3)}")
print(f"Total concatenated: {len(all_adjudications)}")
print(f"Columns: {list(all_adjudications.columns)}")
print()

adjudicated_pairs = set(zip(all_adjudications['Subject ID'], all_adjudications['Phenotype']))

missing_pairs = pd.Series(
    list(zip(pipeline_missing_cohort['Subject ID'], pipeline_missing_cohort['Phenotype'])),
    index=pipeline_missing_cohort.index
)
needs_adjudication = pipeline_missing_cohort[~missing_pairs.isin(adjudicated_pairs)]

print(f"Cells checked: {len(pipeline_missing_cohort)}")
print(f"Already present in adjudication files: {len(pipeline_missing_cohort) - len(needs_adjudication)}")
print(f"NOT in adjudication files -- needs new adjudication: {len(needs_adjudication)}")
print()
print(needs_adjudication.groupby('Phenotype').size().sort_values(ascending=False))
print()
print(needs_adjudication[['Subject ID', 'Phenotype']].sort_values(['Phenotype', 'Subject ID']).to_string(index=False))

In [0]:
check_full = pipeline_missing_cohort.copy()
check_full['pair'] = list(zip(check_full['Subject ID'], check_full['Phenotype']))
check_full['already_adjudicated'] = check_full['pair'].isin(adjudicated_pairs)

summary = check_full.groupby('Phenotype').agg(
    missing_from_manual_score=('pair', 'count'),
    already_adjudicated=('already_adjudicated', 'sum')
)
summary['still_needs_adjudication'] = summary['missing_from_manual_score'] - summary['already_adjudicated']

print(summary)
print()
print(f"Column totals: {summary['missing_from_manual_score'].sum()}, {summary['already_adjudicated'].sum()}, {summary['still_needs_adjudication'].sum()}")

In [0]:
final_needs_adjudication = needs_adjudication.merge(
    combined_df[['Subject ID', 'Phenotype', 'Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']],
    on=['Subject ID', 'Phenotype'],
    how='left'
).merge(
    notes_enrolled4[['Subject ID', 'NOTE_TEXT']],
    on='Subject ID',
    how='left'
)

print(f"Rows: {len(final_needs_adjudication)}")
print(f"Rows missing note text (join problem if > 0): {final_needs_adjudication['NOTE_TEXT'].isna().sum()}")
print()
print(final_needs_adjudication[['Subject ID', 'Phenotype', 'Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']].sort_values(['Phenotype', 'Subject ID']).to_string(index=False))

In [0]:
final_needs_adjudication = final_needs_adjudication.merge(
    results_GPT_full_df[['Subject ID', 'Phenotype', 'Justification', 'Quote']],
    on=['Subject ID', 'Phenotype'],
    how='left'
)

print(f"Rows: {len(final_needs_adjudication)}")
print(f"Rows missing Justification after merge (join problem if > 0): {final_needs_adjudication['Justification'].isna().sum()}")
print()

display(final_needs_adjudication)

In [0]:
print(list(final_needs_adjudication.columns))

In [0]:
current_terms = set(checklist_filtered['Term from checklist'].unique())

pairs_to_check = [
    ('Autism Spectrum Disorder (ASD)', 'Autism Spectrum Disorder (ASD'),
    ('Diabetes Mellitus Type I (insulin dependent)', 'Diabetes Mellitus Type I (insulin dependent'),
    ('Diabetes Mellitus Type II (non-insulin dependent)', 'Diabetes Mellitus Type II (non-insulin dependent'),
    ('Double Outlet Right Ventricle (DORV)', 'Double Outlet Right Ventricle (DORV'),
    ('Hypertrophic cardiomyopathy (HCM)', 'Hypertrophic cardiomyopathy (HCM'),
    ('Tetralogy of Fallot (TOF)', 'Tetralogy of Fallot (TOF'),
]

print(f"Unique terms in checklist_filtered right now: {len(current_terms)}")
print()
for closed, unclosed in pairs_to_check:
    print(f"closed form   {'PRESENT' if closed in current_terms else 'absent  '}  {closed!r}")
    print(f"unclosed form {'PRESENT' if unclosed in current_terms else 'absent  '}  {unclosed!r}")
    print()

In [0]:
dropped_355 = pd.read_csv("/Workspace/Users/user/user/manual_score_dropped_positives.csv")

cohort_ids = set(combined_df['Subject ID'])
dropped_cohort = dropped_355[dropped_355['Subject ID'].isin(cohort_ids)]

pairs = pd.Series(
    list(zip(dropped_cohort['Subject ID'], dropped_cohort['Phenotype'])),
    index=dropped_cohort.index
)
not_in_adjudication = dropped_cohort[~pairs.isin(adjudicated_pairs)]

print(f"Raw-source-confirmed cells, restricted to cohort: {len(dropped_cohort)}")
print(f"Not present in adjudication files: {len(not_in_adjudication)}")
print()
print(not_in_adjudication.groupby('Phenotype').size().sort_values(ascending=False))

In [0]:
to_append = not_in_adjudication.merge(
    combined_df[['Subject ID', 'Phenotype', 'Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']],
    on=['Subject ID', 'Phenotype'], how='left'
).merge(
    notes_enrolled4[['Subject ID', 'NOTE_TEXT']],
    on='Subject ID', how='left'
).merge(
    results_GPT_full_df[['Subject ID', 'Phenotype', 'Justification', 'Quote']],
    on=['Subject ID', 'Phenotype'], how='left'
)

print(f"Rows: {len(to_append)}")
print(f"Missing NOTE_TEXT: {to_append['NOTE_TEXT'].isna().sum()}")
print(f"Missing Justification: {to_append['Justification'].isna().sum()}")
print()
cols_show = ['Subject ID', 'Phenotype', 'Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
print(to_append[cols_show].sort_values(['Phenotype', 'Subject ID']).to_string(index=False))

In [0]:
group3_fresh = pd.read_csv("/Workspace/Users/user/user/Dustin_Adjudications_Unscored.csv")
dustin_schema = list(group3_fresh.columns)

dustin_new_rows = to_append[dustin_schema].copy()

dustin_combined = pd.concat([group3_fresh, dustin_new_rows], ignore_index=True)

dup_check = dustin_combined.duplicated(subset=['Subject ID', 'Phenotype']).sum()

print(f"Dustin's original rows: {len(group3_fresh)}")
print(f"New rows being added: {len(dustin_new_rows)}")
print(f"Combined total: {len(dustin_combined)}")
print(f"Duplicate Subject ID x Phenotype pairs after combining: {dup_check}")
print(f"Schema: {dustin_schema}")

out_path = "/Workspace/Users/user/user/Dustin_Adjudications_Unscored_updated.csv"
dustin_combined.to_csv(out_path, index=False)
print()
print(f"Saved to {out_path}")

In [0]:
import pandas as pd

updated_path = (
    "/Workspace/Users/user/user/Dustin_Adjudications_Unscored_updated.csv"
)

original_path = (
    "/Workspace/Users/user/user/Dustin_Adjudications_Unscored.csv"
)

dustin_updated_unscored = pd.read_csv(updated_path)
dustin_original_unscored = pd.read_csv(original_path)

key_columns = ["Subject ID", "Phenotype"]

# Confirm the comparison columns exist in both files
for column in key_columns:
    if column not in dustin_updated_unscored.columns:
        raise KeyError(f"{column!r} is missing from the updated file.")

    if column not in dustin_original_unscored.columns:
        raise KeyError(f"{column!r} is missing from the original file.")

# Identify rows in the updated file that are absent from the original
original_pairs = (
    dustin_original_unscored[key_columns]
    .drop_duplicates()
    .assign(_present_in_original=True)
)

dustin_added_rows = dustin_updated_unscored.merge(
    original_pairs,
    on=key_columns,
    how="left",
    validate="many_to_one"
)

dustin_added_rows = (
    dustin_added_rows[
        dustin_added_rows["_present_in_original"].isna()
    ]
    .drop(columns="_present_in_original")
    .reset_index(drop=True)
)

print(
    "Rows in updated file but not original file:",
    f"{len(dustin_added_rows):,}"
)

display(dustin_added_rows)

In [0]:
extras_path = (
    "/Workspace/Users/user/user/"
    "Dustin_Adjudications_Extras_Unscored.csv"
)

dustin_added_rows.to_csv(
    extras_path,
    index=False
)

print(f"Saved {len(dustin_added_rows):,} rows to:")
print(extras_path)